# Lab 10: GPT Generation

**Module 10** | Read `notes/10-gpt-causal-lms.pdf` before running this notebook.

Load a small causal language model and compare greedy decoding, temperature sampling, and nucleus (top-p) sampling.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## GPT-2 text generation

Causal language models predict the next token given all previous tokens. At inference time we **decode** autoregressively: sample or select one token, append it, and repeat. Two knobs dominate output diversity:

- **Temperature** scales logits before softmax. Low temperature (0.3) sharpens the distribution toward likely tokens; high temperature (1.2) flattens it and increases surprise.
- **Top-p (nucleus) sampling** keeps the smallest set of tokens whose cumulative probability exceeds `p` (here 0.9) and samples only within that set.

We also compare **greedy decoding** (always pick the argmax token) against stochastic sampling at three temperatures.


### Step 1: Load GPT-2 and print model info

GPT-2 small (~124M parameters) fits most lab machines. If memory is tight the cell below falls back to DistilGPT-2, which is smaller but follows the same generation API.


In [ ]:
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path("..").resolve()
PROMPT = "The future of machine learning is"
TEMPERATURES = [0.3, 0.7, 1.2]
TOP_P = 0.9
MAX_NEW_TOKENS = 60

try:
    model_name = "gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
except Exception:
    model_name = "distilgpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.to(device)
model.eval()

vocab_size = tokenizer.vocab_size
total_params = sum(p.numel() for p in model.parameters())
config = model.config

print("Model info")
print(f"  Name:              {model_name}")
print(f"  Device:            {device}")
print(f"  Vocab size:        {vocab_size:,}")
print(f"  Parameters:        {total_params:,}")
print(f"  Hidden size:       {config.n_embd}")
print(f"  Layers:            {config.n_layer}")
print(f"  Attention heads:   {config.n_head}")
print(f"  Max context:       {config.n_positions} tokens")
print(f"\nPrompt: {PROMPT!r}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")


### Step 2: Greedy decoding baseline

Greedy decoding (`do_sample=False`) always selects the highest-probability next token. Output is deterministic and often repetitive, but useful as a baseline before sampling.


In [ ]:
inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
prompt_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    greedy_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

greedy_text = tokenizer.decode(greedy_ids[0], skip_special_tokens=True)
greedy_new_tokens = greedy_ids.shape[1] - prompt_len

print("Greedy decoding (deterministic argmax)")
print(f"  New tokens generated: {greedy_new_tokens}")
print(f"  Full text:\n{greedy_text}")


### Step 3: Nucleus sampling at three temperatures

For each temperature we run nucleus sampling with the same prompt and seed so differences reflect decoding settings, not random initialization. We print the full generated text and token count for side-by-side comparison.


In [ ]:
rows = []

for temp in TEMPERATURES:
    torch.manual_seed(42)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(42)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=temp,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    new_tokens = output_ids.shape[1] - prompt_len
    rows.append({
        "method": f"sample (T={temp}, top_p={TOP_P})",
        "temperature": temp,
        "top_p": TOP_P,
        "new_tokens": new_tokens,
        "generated_text": text,
    })

    print(f"\n{'=' * 60}")
    print(f"Temperature = {temp} | top_p = {TOP_P} | new tokens = {new_tokens}")
    print(f"{'=' * 60}")
    print(text)

df = pd.DataFrame(rows)
print("\nSummary table")
print(df[["temperature", "top_p", "new_tokens"]].to_string(index=False))


### Step 4: Compare decoding strategies

Greedy output is usually the most predictable. Low temperature sampling stays focused but allows slight variation. High temperature introduces diverse word choices and occasional digressions. Top-p prevents sampling from extremely unlikely tokens while still allowing multiple continuations.


In [ ]:
comparison = [
    {
        "method": "greedy (argmax)",
        "temperature": 0.0,
        "top_p": 1.0,
        "new_tokens": greedy_new_tokens,
        "generated_text": greedy_text,
    },
    *rows,
]

compare_df = pd.DataFrame(comparison)
print("All decoding runs")
for i, row in compare_df.iterrows():
    print(f"\n--- {row['method']} ({row['new_tokens']} new tokens) ---")
    print(row["generated_text"])

print("\nTakeaway: compare repetition, specificity, and tone across methods.")
